# Entrenamiento YOLO26 — Comida Chilena
**Autor:** Felipe Paimilla — Ingeniero Civil Informático

Notebook para entrenar un detector de comida chilena usando YOLO26, el modelo más reciente de Ultralytics (2025).
El dataset `Comida-Chilena-7` se descarga directo desde Roboflow.

**Antes de empezar:** activar GPU en `Entorno de ejecución → Cambiar tipo → T4 GPU`

---
**Estructura del dataset:**
```
Comida-Chilena-7/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── data.yaml
```

## 1. Verificar GPU y recursos

In [ ]:
import subprocess, psutil, shutil

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'GPU no detectada — activa T4 en el entorno de ejecución')

ram  = psutil.virtual_memory()
disk = shutil.disk_usage('/')
print(f'RAM total:      {ram.total/1e9:.1f} GB')
print(f'RAM disponible: {ram.available/1e9:.1f} GB')
print(f'Disco libre:    {disk.free/1e9:.1f} GB')

## 2. Instalar dependencias

In [ ]:
!pip install -U ultralytics roboflow -q

import ultralytics
ultralytics.checks()
print(f'Ultralytics {ultralytics.__version__}')

## 3. Descargar dataset desde Roboflow

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key='6mky4ViTPFpqIFEqG3hS')
project = rf.workspace('nutriplato').project('comida-chilena-mbfky')
version = project.version(7)
dataset = version.download('yolo26')

DATASET_PATH = dataset.location
print(f'Dataset en: {DATASET_PATH}')

## 4. Corregir rutas del data.yaml

Roboflow a veces escribe rutas absolutas que no funcionan en Colab. Esta celda las corrige.

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path(DATASET_PATH) / 'data.yaml'

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = DATASET_PATH
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'path:  {cfg["path"]}')
print(f'train: {cfg["train"]}')
print(f'val:   {cfg["val"]}')
print(f'test:  {cfg["test"]}')
print(f'clases ({cfg["nc"]}): {cfg["names"]}')

## 5. Resumen del dataset

In [ ]:
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
BASE = Path(DATASET_PATH)

total = 0
for split in ['train', 'valid', 'test']:
    imgs = list((BASE / split / 'images').glob('*')) if (BASE / split / 'images').exists() else []
    imgs = [f for f in imgs if f.suffix.lower() in EXTS]
    lbls = list((BASE / split / 'labels').glob('*.txt')) if (BASE / split / 'labels').exists() else []
    print(f'{split:6s}: {len(imgs):6,} imágenes | {len(lbls):6,} etiquetas')
    total += len(imgs)
print(f'TOTAL:  {total:,} imágenes')

## 6. Entrenamiento

Variantes disponibles según recursos:

| Modelo     | mAP@50-95 | GPU T4 (ms) | Params |
|------------|-----------|-------------|--------|
| yolo26n.pt | 40.1      | 1.7         | 2.4M   |
| yolo26s.pt | 47.8      | 2.5         | 9.5M   |
| yolo26m.pt | 52.5      | 4.7         | 20.4M  |
| yolo26l.pt | 54.3      | —           | ~40M   |
| yolo26x.pt | 56.8      | —           | ~60M   |

Para este dataset se usa `yolo26m`. Si hay error de memoria, bajar `BATCH` a 8 o cambiar a `yolo26s`.

In [ ]:
from ultralytics import YOLO

MODEL  = 'yolo26m.pt'
DATA   = str(Path(DATASET_PATH) / 'data.yaml')
EPOCHS = 100
IMGSZ  = 640
BATCH  = 16

model = YOLO(MODEL)

results = model.train(
    data        = DATA,
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    workers     = 4,
    optimizer   = 'auto',   # selecciona MuSGD automáticamente
    patience    = 30,
    save        = True,
    save_period = 10,
    plots       = True,
    project     = 'runs/comida_chilena',
    name        = 'yolo26m',
    device      = 0,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
)

BEST = 'runs/comida_chilena/yolo26m/weights/best.pt'
print(f'Listo. Pesos en: {BEST}')

## 7. Validación

In [ ]:
from ultralytics import YOLO

model   = YOLO(BEST)
metrics = model.val(data=DATA, imgsz=IMGSZ, device=0, plots=True)

print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 8. Visualizar métricas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

RUN = Path('runs/comida_chilena/yolo26m')

for fname, title in [
    ('results.png',                      'Métricas de entrenamiento'),
    ('confusion_matrix_normalized.png',  'Matriz de confusión'),
]:
    p = RUN / fname
    if p.exists():
        plt.figure(figsize=(14, 6))
        plt.imshow(mpimg.imread(p))
        plt.axis('off')
        plt.title(title)
        plt.tight_layout()
        plt.show()

## 9. Inferencia sobre imágenes de test

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

model = YOLO(BEST)

model.predict(
    source  = str(Path(DATASET_PATH) / 'test' / 'images'),
    imgsz   = IMGSZ,
    conf    = 0.25,
    save    = True,
    project = 'runs/comida_chilena',
    name    = 'test_pred',
    device  = 0,
)

# Mostrar primeras 4 imágenes
pred_dir = Path('runs/comida_chilena/test_pred')
imgs = sorted(pred_dir.glob('*.jpg'))[:4] + sorted(pred_dir.glob('*.png'))[:4]
imgs = imgs[:4]

if imgs:
    fig, axes = plt.subplots(1, len(imgs), figsize=(16, 5))
    if len(imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, imgs):
        ax.imshow(mpimg.imread(p))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 10. Exportar modelo

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST)

# ONNX es el formato más portable
model.export(format='onnx', imgsz=IMGSZ, dynamic=True, simplify=True)

# Otras opciones:
# model.export(format='tflite')   # TensorFlow Lite
# model.export(format='coreml')   # iOS / macOS
# model.export(format='engine')   # TensorRT

## 11. Guardar en Google Drive

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

SRC = Path('runs/comida_chilena')
DST = Path('/content/drive/MyDrive/comida_chilena_yolo26')

shutil.copytree(SRC, DST, dirs_exist_ok=True)
print(f'Guardado en: {DST}')
print(f'  best.pt  → {DST}/yolo26m/weights/best.pt')
print(f'  results  → {DST}/yolo26m/results.png')

---
## Notas

**El modelo no converge:** bajar `lr0` a `0.005` o subir `warmup_epochs` a `5`.

**Overfitting** (train loss baja, val sube): reducir épocas o subir `dropout`.

**OOM (out of memory):** bajar `BATCH` a `8` o usar `yolo26s`.

**Reanudar entrenamiento cortado:**
```python
model = YOLO('runs/comida_chilena/yolo26m/weights/last.pt')
model.train(resume=True)
```